# Taxi Trip Fetching

In this notebook we fetch the taxi trip data provided by the city of chicago: https://data.cityofchicago.org/Transportation/Taxi-Trips-2024-/ajtu-isnz/about_data (downloaded on 05.05.2026).

In [21]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Import packages                         #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

import pandas as pd
import numpy as np

from sodapy import Socrata
import requests

import time

In [ ]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Fetch data info                         #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

URL = "https://data.cityofchicago.org/resource/ajtu-isnz.json"
params = {
    "$select": "count(*) AS n",
    "$where": "trip_start_timestamp >= '2025-01-01T00:00:00' "
              "AND trip_start_timestamp < '2026-01-01T00:00:00'",
}
print(requests.get(URL, params=params, timeout=120).json())

In [ ]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Fetch data                              #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

API_ENDPOINT = "https://data.cityofchicago.org/resource/ajtu-isnz.csv"
LIMIT = 50000
MAX_RETRIES = 5
OUTPUT_FILE = "CHICAGO_TAXI_TRIPS_2025.csv"

def fetch_page(offset):
    params = {
        "$where": "trip_start_timestamp >= '2025-01-01T00:00:00' "
                  "AND trip_start_timestamp < '2026-01-01T00:00:00'",
        "$order": "trip_start_timestamp",
        "$limit": LIMIT,
        "$offset": offset,
    }
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            with requests.get(API_ENDPOINT, params=params, timeout=120, stream=True) as r:
                r.raise_for_status()
                return r.content
        except (requests.exceptions.ChunkedEncodingError,
                requests.exceptions.ConnectionError,
                requests.exceptions.Timeout) as e:
            print(f"  Attempt {attempt}/{MAX_RETRIES} failed: {e}")
            if attempt == MAX_RETRIES:
                raise
            time.sleep(2 ** attempt)

offset = 0
first_page = True
with open(OUTPUT_FILE, "wb") as f:
    while True:
        print(f"Fetching rows {offset}–{offset + LIMIT - 1}...")
        content = fetch_page(offset)

        lines = content.splitlines(keepends=True)
        if first_page:
            f.writelines(lines)
            first_page = False
        else:
            f.writelines(lines[1:])  # skip header on subsequent pages

        row_count = len(lines) - 1  # exclude header
        print(f"  Got {row_count} rows.")
        if row_count < LIMIT:
            break
        offset += LIMIT

print("Download complete.")